# 金融資料探勘 - 作業一：資料處理流程檔
**學號：** 411336062
**負責年度：** 2022年

### 處理流程說明：
1. **讀取資料**：匯入 2022 年大盤收盤價原始檔。
2. **檔名建構**：運用 `strftime` 生成對應的 `OptionsDaily_yyyy_mm_dd.csv`。
3. **契約與到期日**：撰寫函數自動抓取每個月的「第三個星期三」作為結算日。
4. **到期天數計算**：以日曆日計算 `ContractExpiryDate` 與 `Date` 的差值。
5. **動態利率寫入**：依據 2022 年央行四次升息決議（3/18, 6/17, 9/23, 12/16），動態判斷並寫入對應的重貼現率 (1.125% ~ 1.75%)。
6. **格式校準與輸出**：依照官方參考範例格式，將日期轉為 `YYYY-MM-DD`，排序 7 個指定欄位後匯出 Excel。

In [6]:
import pandas as pd
import datetime

print("開始處理 2022 年度索引檔 (依官方 PDF 參考範例格式)...")

# ==========================================
# 1. 讀取 2022 年的原始資料 (改為讀取 Excel 檔)
# ==========================================
# 已經將檔名與副檔名更改為 2022資料.xlsx
file_path = '2022資料.xlsx'
df = pd.read_excel(file_path) # 改用 read_excel 來讀取 xlsx 檔案
df['Date'] = pd.to_datetime(df['Date'])

# ==========================================
# 2. 建立每日檔名 File
# ==========================================
df['File'] = df['Date'].apply(lambda x: f"OptionsDaily_{x.strftime('%Y_%m_%d')}.csv")

# ==========================================
# 3. 建立近月契約 Contract 與 ContractExpiryDate
# ==========================================
def get_third_wednesday(year, month):
    first_day = datetime.date(year, month, 1)
    days_to_wednesday = (2 - first_day.weekday()) % 7
    return first_day + datetime.timedelta(days=days_to_wednesday + 14)

def get_contract_info(date_obj):
    tw = get_third_wednesday(date_obj.year, date_obj.month)
    if date_obj.date() <= tw:
        expiry = tw
    else:
        if date_obj.month == 12:
            expiry = get_third_wednesday(date_obj.year + 1, 1)
        else:
            expiry = get_third_wednesday(date_obj.year, date_obj.month + 1)

    contract = expiry.strftime('%Y%m')
    return pd.Series([contract, expiry])

df[['Contract', 'ContractExpiryDate']] = df['Date'].apply(get_contract_info)

# ==========================================
# 4. 計算 Maturity (距離到期天數)
# ==========================================
df['ContractExpiryDate'] = pd.to_datetime(df['ContractExpiryDate'])
df['Maturity'] = (df['ContractExpiryDate'] - df['Date']).dt.days

# ==========================================
# 5. 補入 Rf (2022年 台灣央行升息動態判斷)
# ==========================================
def get_rf_2022(date_val):
    if date_val < pd.to_datetime('2022-03-18'):
        return 0.01125
    elif date_val < pd.to_datetime('2022-06-17'):
        return 0.01375
    elif date_val < pd.to_datetime('2022-09-23'):
        return 0.01500
    elif date_val < pd.to_datetime('2022-12-16'):
        return 0.01625
    else:
        return 0.01750

df['Rf'] = df['Date'].apply(get_rf_2022)

# ==========================================
# 6. 整理欄位與日期格式
# ==========================================
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df['ContractExpiryDate'] = df['ContractExpiryDate'].dt.strftime('%Y-%m-%d')
final_cols = ['Date', 'File', 'S0', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf']
df_final = df[final_cols]

# ==========================================
# 7. 輸出最終 Excel 檔案
# ==========================================
output_filename = 'Index_411336062_2022.xlsx'
df_final.to_excel(output_filename, index=False, engine='openpyxl')

print(f"✅ 成功產出：{output_filename}")
print("\n前 5 筆資料預覽：")
print(df_final.head(5))

開始處理 2022 年度索引檔 (依官方 PDF 參考範例格式)...
✅ 成功產出：Index_411336062_2022.xlsx

前 5 筆資料預覽：
         Date                         File            S0 Contract  \
0  2022-01-03  OptionsDaily_2022_01_03.csv  18270.509766   202201   
1  2022-01-04  OptionsDaily_2022_01_04.csv  18526.349609   202201   
2  2022-01-05  OptionsDaily_2022_01_05.csv  18499.960938   202201   
3  2022-01-06  OptionsDaily_2022_01_06.csv  18367.919922   202201   
4  2022-01-07  OptionsDaily_2022_01_07.csv  18169.759766   202201   

  ContractExpiryDate  Maturity       Rf  
0         2022-01-19        16  0.01125  
1         2022-01-19        15  0.01125  
2         2022-01-19        14  0.01125  
3         2022-01-19        13  0.01125  
4         2022-01-19        12  0.01125  
